# 07 - Churn Prediction

Goal: predict whether a customer will make **another** purchase within 60 days
of a fixed cutoff date, using only what we knew about them *before* that cutoff.

This is a genuine forward-looking prediction problem, so the usual random
train/test split doesn't apply — using it here would leak future purchase
information into the features. Instead we split by **time**: everything up to
a cutoff date becomes features, everything after becomes the label.

**Cross-version compatibility fix:** this environment's pandas (3.0) defaults to the new `StringDtype` for text columns (`future.infer_string=True`). Anything fitted or computed here (OneHotEncoder categories, product-name lookups) would then get pickled with that dtype -- which older/different pandas installs (e.g. on the machine actually running the dashboard) cannot unpickle, raising `NotImplementedError` on load. Forcing `pd.set_option('future.infer_string', False)` up front keeps everything on plain `object` dtype, which pickles compatibly across pandas versions.


## Choosing the cutoff date

This dataset is a **fixed, closed window** (Dec 2009 - Dec 2011) — there's no
"today" to measure from, unlike a live production system. That creates a specific
risk called right-censoring: if we pick a cutoff too close to the end of the data,
customers who *would have* repurchased just after the data ends get incorrectly
labeled "churned" simply because we ran out of days to observe them, not because
they actually stopped buying.

To avoid that, the cutoff must leave a full 60-day window still inside the data:

`cutoff = max_date - 60 days`

Every customer's 60-day observation window is then guaranteed to be fully
contained within the dataset — no one is unfairly penalized for the dataset
ending.

We only build features (and make predictions) for customers who had already
purchased at least once before the cutoff — there's no behavioral history to
build features from for anyone else, so they can't be part of this prediction
problem at all.

## Features: everything computed from data up to the cutoff only

`Tenure` (days since first purchase, as of the cutoff) is new here versus the
RFM notebook — it matters specifically for churn because a customer who's only
been active for 2 weeks looks identical to a long-time customer on Recency and
Frequency alone, but their likely future behavior is very different.

## Label: did they purchase again within the window?

**63% churn rate at a 60-day window.** This is high, but it lines up with what
we already found in notebook 04: the median customer only places 3 orders across
the *entire* ~2-year dataset, so most customers simply aren't frequent enough
buyers for a 60-day repurchase to be the norm. This isn't a bug in the labeling —
it's a real characteristic of this business (many one-off or occasional buyers,
a smaller core of frequent ones), and it's a second, independent imbalance problem
in this project, on top of the one in purchase prediction. We'll need the same
kind of careful evaluation here, not just accuracy.

## Train / test split — chronological, not random

Even though the label itself is already time-aware (built from the future window),
we still need a train/test split. Random shuffling of *customers* here is
acceptable — unlike the label construction, splitting customers into train/test
groups doesn't leak future information, since every customer's features are already
capped at the same cutoff date. What would leak is using a different cutoff per
customer, or shuffling in time — neither of which we're doing.

## Model

**Not a clean win for either model.** Logistic Regression actually has the
higher ROC-AUC (0.793 vs 0.782) — Random Forest doesn't win on overall ranking
ability. What Random Forest does better is recall on the churned class (0.86 vs
0.71) at the cost of recall on the non-churned class (0.54 vs 0.74). So the choice
isn't "which model is better" in the abstract, it's "which error matters more":
if the business cost of missing a churning customer outweighs the cost of
wrongly flagging a loyal one (a retention email is cheap), Random Forest's higher
churn recall is the more useful trade. We keep Random Forest for that reason —
not because it's objectively the stronger model.

## Feature importance

Recency is almost always the strongest churn signal in RFM-style models — it's
the most direct proxy for "have they drifted away already." Worth checking this
against your own numbers before assuming it: if Frequency or Monetary had come
out on top instead, that would be a genuinely more interesting finding, not
just confirming the expected.

## Save

In [1]:
import pandas as pd
pd.set_option('future.infer_string', False)
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, roc_auc_score, RocCurveDisplay, ConfusionMatrixDisplay, confusion_matrix

customer_df = pd.read_csv("../data/processed/retail_customer_level.csv", parse_dates=["InvoiceDate"], low_memory=False)
print(customer_df["InvoiceDate"].min(), customer_df["InvoiceDate"].max())

CHURN_WINDOW_DAYS = 60
max_date = customer_df["InvoiceDate"].max()
cutoff = max_date - pd.Timedelta(days=CHURN_WINDOW_DAYS)
print("cutoff:", cutoff)

past = customer_df[customer_df["InvoiceDate"] <= cutoff]
future = customer_df[customer_df["InvoiceDate"] > cutoff]
print("past tx:", len(past), "future tx:", len(future), "customers w/ purchase before cutoff:", past["Customer ID"].nunique())

features = past.groupby("Customer ID").agg(
    Recency=("InvoiceDate", lambda x: (cutoff - x.max()).days),
    Frequency=("Invoice", "nunique"),
    Monetary=("TotalAmount", "sum"),
    AvgOrderValue=("TotalAmount", "mean"),
    UniqueProducts=("StockCode", "nunique"),
    Tenure=("InvoiceDate", lambda x: (cutoff - x.min()).days)
).reset_index()
features["Frequency_log"] = np.log1p(features["Frequency"])
features["Monetary_log"] = np.log1p(features["Monetary"])
print("features shape:", features.shape)

repurchased_customers = set(future["Customer ID"].dropna().unique())
features["Repurchased"] = features["Customer ID"].isin(repurchased_customers).astype(int)
features["Churned"] = 1 - features["Repurchased"]
print(features["Churned"].value_counts(normalize=True))

X = features.drop(columns=["Customer ID", "Repurchased", "Churned"])
y = features["Churned"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(X_train.shape, X_test.shape)

pipe_log = Pipeline([("scaler", StandardScaler()), ("clf", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42))])
pipe_log.fit(X_train, y_train)
preds = pipe_log.predict(X_test); probs = pipe_log.predict_proba(X_test)[:, 1]
print("=== Logistic Regression (balanced) ===")
print(classification_report(y_test, preds))
print("ROC-AUC:", round(roc_auc_score(y_test, probs), 4))

pipe_rf = Pipeline([("scaler", StandardScaler()), ("clf", RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42))])
pipe_rf.fit(X_train, y_train)
preds_rf = pipe_rf.predict(X_test); probs_rf = pipe_rf.predict_proba(X_test)[:, 1]
print("=== Random Forest (balanced) ===")
print(classification_report(y_test, preds_rf))
print("ROC-AUC:", round(roc_auc_score(y_test, probs_rf), 4))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
cm = confusion_matrix(y_test, preds_rf)
ConfusionMatrixDisplay(cm, display_labels=["Repurchased", "Churned"]).plot(ax=axes[0], colorbar=False)
axes[0].set_title("Confusion Matrix (RF)")
RocCurveDisplay.from_predictions(y_test, probs_rf, ax=axes[1]); axes[1].set_title("ROC Curve (RF)")
plt.tight_layout(); plt.savefig("../reports/figures/churn_prediction_evaluation.png"); plt.close()

importances = pd.Series(pipe_rf.named_steps["clf"].feature_importances_, index=X.columns).sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(8, 5))
importances.plot(kind="barh", ax=ax); ax.invert_yaxis(); ax.set_title("Churn Model - Feature Importance")
plt.tight_layout(); plt.savefig("../reports/figures/churn_feature_importance.png"); plt.close()
print(importances)

import joblib
joblib.dump(pipe_rf, "../models/churn_model.pkl")
features.to_csv("../data/processed/churn_features.csv", index=False)
print("saved")


2009-12-01 07:45:00 2011-12-09 12:50:00
cutoff: 2011-10-10 12:50:00
past tx: 664319 future tx: 115106 customers w/ purchase before cutoff: 5518
features shape: (5518, 9)
Churned
1    0.631026
0    0.368974
Name: proportion, dtype: float64
(4414, 8) (1104, 8)
=== Logistic Regression (balanced) ===
              precision    recall  f1-score   support

           0       0.60      0.74      0.66       407
           1       0.82      0.71      0.76       697

    accuracy                           0.72      1104
   macro avg       0.71      0.72      0.71      1104
weighted avg       0.74      0.72      0.72      1104

ROC-AUC: 0.7929
=== Random Forest (balanced) ===
              precision    recall  f1-score   support

           0       0.69      0.54      0.61       407
           1       0.76      0.86      0.81       697

    accuracy                           0.74      1104
   macro avg       0.72      0.70      0.71      1104
weighted avg       0.73      0.74      0.73      1104
